# Team 404 — Task 2: Used Car Price Prediction

Upload **dataset-2.csv**, then choose **Runtime → Run all**. Uses only the supplied data. This preparation notebook covers cleaning, feature engineering, leakage-safe pipelines, cross-validation, three regression models, held-out evaluation, model saving, and new-car predictions.

The official Task 2 starter was not attached; adjust submission naming if required. The competition guidelines prohibit AI tools during the task.

RMSE and MAE are in dollars; lower is better. Higher R² is better. The saved model is trained on development data, leaving test results independent.

## 1. Imports and settings

In [ ]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from IPython.display import display
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
SEED = 404
OUTPUT_DIR = Path('task2_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)


## 2. Load dataset-2.csv

In [ ]:
DATA_PATH = Path('dataset-2.csv')
if not DATA_PATH.exists():
    alternatives = [Path('/content/dataset-2.csv'), Path('upload/dataset-2.csv')]
    DATA_PATH = next((p for p in alternatives if p.exists()), DATA_PATH)
if not DATA_PATH.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_names = [name for name in uploaded if name.lower().endswith('.csv')]
        if 'dataset-2.csv' in csv_names:
            DATA_PATH = Path('dataset-2.csv')
        elif len(csv_names) == 1:
            DATA_PATH = Path(csv_names[0])
        else:
            raise ValueError('Upload dataset-2.csv only, or set DATA_PATH explicitly.')
    except ImportError as exc:
        raise FileNotFoundError('Place dataset-2.csv beside this code file.') from exc
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()
if 'price' not in df.columns:
    raise ValueError('Training data must contain the price column.')
print('Dataset shape:', df.shape)
display(df.head())
display(df.isna().sum().rename('Missing values').to_frame())


## 3. Clean target and duplicate rows

In [ ]:
def numeric_text(values):
    return pd.to_numeric(values.astype(str).str.replace(r'[^0-9.\-]', '', regex=True), errors='coerce')
df['price'] = numeric_text(df['price'])
before = len(df)
df = df.dropna(subset=['price'])
df = df.loc[np.isfinite(df['price']) & (df['price'] > 0)].drop_duplicates().reset_index(drop=True)
print('Removed invalid target / duplicate rows:', before - len(df))
print('Usable rows:', len(df))
display(df['price'].describe().to_frame())


## 4. Exploratory analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['price'], bins=40, edgecolor='white')
axes[0].set(title='Car price distribution', xlabel='Price ($)', ylabel='Cars')
axes[1].hist(np.log1p(df['price']), bins=40, edgecolor='white')
axes[1].set(title='Log price distribution', xlabel='log(1 + price)', ylabel='Cars')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'price_distribution.png', dpi=150)
plt.show()


## 5. Feature engineering

In [ ]:
def make_features(data):
    x = data.drop(columns=['price'], errors='ignore').copy()
    for col in x.select_dtypes(include=['object', 'string']).columns:
        x[col] = x[col].map(lambda v: v.strip() if isinstance(v, str) else v)
        x[col] = x[col].replace({'': np.nan, '–': np.nan, '—': np.nan, '-': np.nan})
    if 'milage' in x:
        x['milage'] = numeric_text(x['milage'])
    if 'model_year' in x:
        x['model_year'] = pd.to_numeric(x['model_year'], errors='coerce')
    if 'engine' in x:
        engine = x['engine'].fillna('').astype(str)
        x['horsepower'] = pd.to_numeric(engine.str.extract(r'(\d+(?:\.\d+)?)\s*HP', flags=re.I)[0], errors='coerce')
        x['engine_liters'] = pd.to_numeric(engine.str.extract(r'(\d+(?:\.\d+)?)\s*L\b', flags=re.I)[0], errors='coerce')
        cylinder = engine.str.extract(r'(\d+)\s*Cylinder', flags=re.I)[0]
        cylinder = cylinder.fillna(engine.str.extract(r'\b[VIW]\s*(\d+)\b', flags=re.I)[0])
        x['cylinders'] = pd.to_numeric(cylinder, errors='coerce')
    if 'transmission' in x:
        trans = x['transmission'].fillna('').astype(str)
        x['transmission_speeds'] = pd.to_numeric(trans.str.extract(r'(\d+)\s*[- ]?Speed', flags=re.I)[0], errors='coerce')
    return x
X = make_features(df)
y = df['price'].astype(float)
FEATURE_COLUMNS = X.columns.tolist()
display(X.head())


## 6. Training / validation / test split

In [ ]:
# The test set stays untouched until final evaluation.
X_development, X_test, y_development, y_test = train_test_split(X, y, test_size=0.20, random_state=SEED)
X_train, X_validation, y_train, y_validation = train_test_split(X_development, y_development, test_size=0.25, random_state=SEED)
train_split = df.loc[X_train.index].copy()
validation_split = df.loc[X_validation.index].copy()
evaluation_split = df.loc[X_test.index].copy()
for name, frame in [('train_split', train_split), ('validation_split', validation_split), ('evaluation_split', evaluation_split)]:
    frame.to_csv(OUTPUT_DIR / (name + '.csv'), index=False)
    print(name, frame.shape)


## 7. Leakage-safe preprocessing

In [ ]:
numeric_columns = X.select_dtypes(include=np.number).columns.tolist()
categorical_columns = [c for c in FEATURE_COLUMNS if c not in numeric_columns]
def make_preprocessor():
    numeric = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
    categorical = Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')), ('encoder', OneHotEncoder(handle_unknown='ignore'))])
    return ColumnTransformer([('numeric', numeric, numeric_columns), ('categorical', categorical, categorical_columns)])
def make_model(regressor):
    # Log target helps with the highly skewed prices. Metrics remain in dollars.
    return TransformedTargetRegressor(regressor=Pipeline([('preprocess', make_preprocessor()), ('model', regressor)]), func=np.log1p, inverse_func=np.expm1)
models = {
    'Ridge': make_model(Ridge(alpha=10.0, solver='lsqr')),
    'Random Forest': make_model(RandomForestRegressor(n_estimators=200, min_samples_leaf=2, max_features=0.8, random_state=SEED, n_jobs=-1)),
    'Extra Trees': make_model(ExtraTreesRegressor(n_estimators=200, min_samples_leaf=2, max_features=0.8, random_state=SEED, n_jobs=-1)),
}


## 8. Evaluation functions and baseline

In [ ]:
def evaluate_predictions(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float).ravel()
    y_pred = np.asarray(y_pred, dtype=float).ravel()
    if len(y_true) != len(y_pred) or not np.isfinite(y_pred).all():
        raise ValueError('Predictions must be finite and match the number of actual values.')
    return {'MAE': float(mean_absolute_error(y_true, y_pred)), 'RMSE': float(np.sqrt(mean_squared_error(y_true, y_pred))), 'R2': float(r2_score(y_true, y_pred))}
baseline = np.full(len(y_validation), y_train.median())
print('Training-median validation baseline:', evaluate_predictions(y_validation, baseline))


## 9. Compare models using training-only cross-validation

In [ ]:
cv = KFold(n_splits=3, shuffle=True, random_state=SEED)
rows = []
for name, model in models.items():
    print('Evaluating:', name)
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring={'mae': 'neg_mean_absolute_error', 'rmse': 'neg_root_mean_squared_error', 'r2': 'r2'}, n_jobs=1, error_score='raise')
    model.fit(X_train, y_train)
    metrics = evaluate_predictions(y_validation, np.maximum(model.predict(X_validation), 0))
    rows.append({'Model': name, 'CV_MAE': -scores['test_mae'].mean(), 'CV_RMSE': -scores['test_rmse'].mean(), 'CV_R2': scores['test_r2'].mean(), **{'Validation_' + k: v for k, v in metrics.items()}})
comparison = pd.DataFrame(rows).sort_values('CV_RMSE').reset_index(drop=True)
display(comparison)
comparison.to_csv(OUTPUT_DIR / 'model_comparison.csv', index=False)
best_model_name = comparison.iloc[0]['Model']
print('Selected by lowest cross-validation RMSE:', best_model_name)


## 10. Refit selected model and evaluate once on held-out test data

In [ ]:
best_model = models[best_model_name]
best_model.fit(X_development, y_development)
test_predictions = np.maximum(best_model.predict(X_test), 0)
test_metrics = evaluate_predictions(y_test, test_predictions)
test_baseline_metrics = evaluate_predictions(y_test, np.full(len(y_test), y_development.median()))
display(pd.DataFrame([{'Model': 'Median baseline', **test_baseline_metrics}, {'Model': best_model_name, **test_metrics}]))
with open(OUTPUT_DIR / 'test_metrics.json', 'w') as f:
    json.dump({'selected_model': best_model_name, 'test_metrics': test_metrics, 'baseline_metrics': test_baseline_metrics}, f, indent=2)
results = evaluation_split.copy()
results['predicted_price'] = test_predictions
results['residual'] = y_test.to_numpy() - test_predictions
results.to_csv(OUTPUT_DIR / 'test_predictions.csv', index=False)


## 11. Diagnostic plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(y_test, test_predictions, alpha=0.5, s=15)
limit = max(y_test.max(), test_predictions.max())
axes[0].plot([0, limit], [0, limit], 'r--')
axes[0].set(title='Actual vs predicted prices', xlabel='Actual price ($)', ylabel='Predicted price ($)')
axes[1].scatter(test_predictions, y_test.to_numpy() - test_predictions, alpha=0.5, s=15)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set(title='Residuals', xlabel='Predicted price ($)', ylabel='Actual - predicted ($)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'test_diagnostics.png', dpi=150)
plt.show()


## 12. Save evaluated model and prediction function

In [ ]:
joblib.dump(best_model, OUTPUT_DIR / 'car_price_model.joblib')
joblib.dump(FEATURE_COLUMNS, OUTPUT_DIR / 'feature_columns.joblib')
def predict_price(new_data):
    """Accept raw car records as a DataFrame, dictionary, or list of dictionaries."""
    if isinstance(new_data, dict):
        new_data = pd.DataFrame([new_data])
    elif not isinstance(new_data, pd.DataFrame):
        new_data = pd.DataFrame(new_data)
    features = make_features(new_data).reindex(columns=FEATURE_COLUMNS)
    return np.maximum(best_model.predict(features), 0)
example = df.drop(columns=['price']).iloc[[0]].copy()
display(example)
print('Example predicted price: $', round(float(predict_price(example)[0]), 2))


## 13. Optional predictions for a separate evaluation CSV

In [ ]:
# Set this path when the organizers supply a separate unlabeled dataset.
NEW_DATA_PATH = None  # Example: '/content/evaluation_data.csv'
if NEW_DATA_PATH is not None:
    new_data = pd.read_csv(NEW_DATA_PATH)
    new_data.columns = new_data.columns.str.strip()
    new_predictions = predict_price(new_data)
    submission = pd.DataFrame({'price': new_predictions}, index=new_data.index)
    submission.to_csv(OUTPUT_DIR / 'submission.csv', index=False)
    print('Saved submission.csv; adapt its column names to the official submission format.')
    if 'price' in new_data:
        actual = numeric_text(new_data['price'])
        valid = actual.notna() & np.isfinite(actual)
        if valid.sum() >= 2:
            print('Separate evaluation metrics:', evaluate_predictions(actual[valid], new_predictions[valid]))
print('All sections finished. Output folder:', OUTPUT_DIR.resolve())
